# Olist EDA — Exploratory Data Analysis

Walks the processed dataset produced by `make ingest`:
1. Load & profile the cleaned Parquet tables
2. Order-status funnel
3. Revenue seasonality
4. Delivery performance
5. Reviews vs delivery experience


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
PROCESSED = Path("../data/processed")

tables = {p.stem: pd.read_parquet(p) for p in sorted(PROCESSED.glob("*.parquet"))}
{name: df.shape for name, df in tables.items()}

## 1. Profile

Quick shape/null overview for every table.

In [ ]:
profile = pd.DataFrame({
    name: {"rows": len(df), "cols": df.shape[1], "null_cells": int(df.isna().sum().sum())}
    for name, df in tables.items()
}).T.sort_values("rows", ascending=False)
profile

## 2. Order-status funnel

In [ ]:
orders = tables["orders"]
status = orders["order_status"].value_counts()
ax = status.plot.barh(figsize=(8, 4), title="Orders by status")
ax.invert_yaxis()
plt.tight_layout(); plt.show()
status

## 3. Revenue seasonality

Monthly delivered revenue from payments joined to orders.

In [ ]:
payments = tables["order_payments"]
rev = (
    orders.query("order_status == 'delivered'")
    .merge(payments.groupby("order_id", as_index=False)["payment_value"].sum(), on="order_id")
    .assign(month=lambda d: d["order_purchase_timestamp"].dt.to_period("M").astype(str))
    .groupby("month", as_index=False)["payment_value"].sum()
)
ax = rev.plot(x="month", y="payment_value", figsize=(10, 4), marker="o",
              title="Monthly delivered revenue", legend=False)
ax.set_ylabel("BRL")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()
rev.tail()

## 4. Delivery performance

In [ ]:
delivered = orders.dropna(subset=["order_delivered_customer_date"]).copy()
delivered["delivery_days"] = (
    delivered["order_delivered_customer_date"] - delivered["order_purchase_timestamp"]
).dt.days
delivered["on_time"] = (
    delivered["order_delivered_customer_date"] <= delivered["order_estimated_delivery_date"]
)

print(f"On-time rate: {delivered['on_time'].mean():.1%}")
ax = delivered["delivery_days"].plot.hist(bins=40, figsize=(9, 4),
                                          title="Delivery time (days)")
plt.tight_layout(); plt.show()
delivered["delivery_days"].describe()

## 5. Do late deliveries hurt reviews?

In [ ]:
reviews = tables["order_reviews"][["order_id", "review_score"]]
joined = delivered.merge(reviews, on="order_id")
by_outcome = joined.groupby("on_time")["review_score"].agg(["mean", "count"])
by_outcome.index = by_outcome.index.map({True: "on_time", False: "late"})
by_outcome

**Takeaway.** Late deliveries correlate with materially lower review
scores — supporting the delivery-SLA KPIs on the dashboard and the
`delivery_vs_estimate_days` measure in the warehouse.

Next: the star schema (`make warehouse`) and the SQL library in
`sql/queries/` operationalise these findings.